# 🎛️ Interactive Hierarchical Clustering Tool
## Adjust cut heights and see results in real-time

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
import ipywidgets as widgets
from IPython.display import display, clear_output
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


## 📥 Load Data

In [2]:
# Load existing data (use optimal_ prefix)
segs = np.load('optimal_segs.npy', allow_pickle=True)
SCM = np.load('optimal_SCM.npy')
params = np.load('optimal_params.npy')

ws, thr, sigma = params
K = len(segs)

# Calculate FPS (video is 4min38s = 278 seconds, total frames = last frame number)
VIDEO_DURATION_SECONDS = 278  # 4min 38s
TOTAL_FRAMES = segs[-1][1]  # Last frame number
FPS = TOTAL_FRAMES / VIDEO_DURATION_SECONDS
print(f'Detected FPS: {FPS:.2f}')

# Perform hierarchical clustering
distance_matrix = 1 - SCM
condensed_dist = squareform(distance_matrix, checks=False)
Z = linkage(condensed_dist, method='ward')

# Get initial cut heights (current method)
def get_cut_height(Z, n_clusters):
    if n_clusters >= len(Z) + 1:
        return 0
    return Z[-(n_clusters-1), 2]

initial_h_activity = get_cut_height(Z, 2)
initial_h_side = get_cut_height(Z, 4)
initial_h_component = get_cut_height(Z, 8)

max_height = Z[:, 2].max()

print(f'✅ Data loaded: {K} atomic segments')
print(f'Initial cut heights:')
print(f'  Activity: {initial_h_activity:.3f} (2 clusters)')
print(f'  Side: {initial_h_side:.3f} (4 clusters)')
print(f'  Component: {initial_h_component:.3f} (8 clusters)')
print(f'Max height: {max_height:.3f}')

Detected FPS: 10.01
✅ Data loaded: 142 atomic segments
Initial cut heights:
  Activity: 5.694 (2 clusters)
  Side: 1.400 (4 clusters)
  Component: 0.231 (8 clusters)
Max height: 5.694


## 🎨 Define Colors

In [3]:
COLORS = {
    'activity': '#E74C3C',
    'side': '#F39C12',
    'component': '#27AE60',
    'atomic': '#3498DB',
    'primary': '#2C3E50'
}

## 🎛️ Interactive Widget

In [4]:
# Create sliders for each level
slider_activity = widgets.FloatSlider(
    value=initial_h_activity,
    min=initial_h_side + 0.1,  # Must be above side level
    max=max_height,
    step=0.01,
    description='Activity h:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px'),
    readout_format='.3f'
)

slider_side = widgets.FloatSlider(
    value=initial_h_side,
    min=initial_h_component + 0.05,  # Must be above component level
    max=initial_h_activity - 0.1,  # Must be below activity level
    step=0.01,
    description='Side h:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px'),
    readout_format='.3f'
)

slider_component = widgets.FloatSlider(
    value=initial_h_component,
    min=0.01,
    max=initial_h_side - 0.05,  # Must be below side level
    step=0.01,
    description='Component h:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px'),
    readout_format='.3f'
)

# Info labels
label_activity = widgets.HTML(value='<b>Clusters:</b> ?')
label_side = widgets.HTML(value='<b>Clusters:</b> ?')
label_component = widgets.HTML(value='<b>Clusters:</b> ?')

# Reset button
reset_button = widgets.Button(
    description='Reset to Initial',
    button_style='warning',
    icon='refresh'
)

# Auto-optimize button
auto_button = widgets.Button(
    description='Auto-Optimize',
    button_style='success',
    icon='magic'
)

print('✅ Widgets created')

✅ Widgets created


## 🔧 Auto-Optimization Function

In [5]:
from sklearn.metrics import silhouette_score

def auto_optimize_cuts(Z, distance_matrix, n_levels=3):
    """
    Automatically determine optimal cut heights using Elbow + Silhouette methods
    """
    print('🔍 Searching for optimal cut heights...')
    
    # Step 1: Find largest jumps in merge heights (Elbow method)
    height_diffs = np.diff(Z[:, 2])
    candidate_idx = np.argsort(height_diffs)[-15:]  # Top 15 candidates
    
    # Step 2: Evaluate each candidate with Silhouette Score
    scores = {}
    for idx in candidate_idx:
        h = Z[idx, 2]
        labels = fcluster(Z, t=h, criterion='distance')
        n_clusters = len(np.unique(labels))
        
        if n_clusters >= 2 and n_clusters <= 20:
            try:
                score = silhouette_score(distance_matrix, labels, metric='precomputed')
                scores[h] = (score, n_clusters)
                print(f'  h={h:.3f} → {n_clusters} clusters, score={score:.3f}')
            except:
                pass
    
    # Step 3: Select top n_levels by score
    sorted_h = sorted(scores.items(), key=lambda x: x[1][0], reverse=True)
    optimal_h = [x[0] for x in sorted_h[:n_levels]]
    optimal_h.sort(reverse=True)  # Sort from high to low
    
    print(f'\n✅ Optimal cuts found:')
    for i, h in enumerate(optimal_h):
        n_clust = scores[h][1]
        score = scores[h][0]
        print(f'  Level {i+1}: h={h:.3f} → {n_clust} clusters (score={score:.3f})')
    
    return optimal_h, scores

print('✅ Auto-optimization function defined')

✅ Auto-optimization function defined


## 📊 Visualization Function

In [6]:
def update_visualization(h_activity, h_side, h_component):
    """
    Update all visualizations based on current cut heights
    """
    # Get cluster labels
    labels_activity = fcluster(Z, t=h_activity, criterion='distance')
    labels_side = fcluster(Z, t=h_side, criterion='distance')
    labels_component = fcluster(Z, t=h_component, criterion='distance')
    
    n_activity = len(np.unique(labels_activity))
    n_side = len(np.unique(labels_side))
    n_component = len(np.unique(labels_component))
    
    # Update labels
    label_activity.value = f'<b>Clusters:</b> {n_activity}'
    label_side.value = f'<b>Clusters:</b> {n_side}'
    label_component.value = f'<b>Clusters:</b> {n_component}'
    
    # Create figure with 3 subplots
    fig = plt.figure(figsize=(24, 16))
    gs = fig.add_gridspec(3, 2, height_ratios=[2, 1, 1], hspace=0.3, wspace=0.3)
    
    # ========== Subplot 1: Dendrogram ==========
    ax1 = fig.add_subplot(gs[0, :])
    
    plt.rcParams['lines.linewidth'] = 2.0
    dendrogram(Z, ax=ax1, color_threshold=0, 
               above_threshold_color=COLORS['primary'], leaf_font_size=7)
    plt.rcParams['lines.linewidth'] = 1.5
    
    # Add cut lines
    ax1.axhline(y=h_activity, color=COLORS['activity'], linestyle='--', linewidth=4, 
                label=f'Activity (h={h_activity:.3f}, n={n_activity})', alpha=0.9, zorder=10)
    ax1.axhline(y=h_side, color=COLORS['side'], linestyle='--', linewidth=4, 
                label=f'Side (h={h_side:.3f}, n={n_side})', alpha=0.9, zorder=10)
    ax1.axhline(y=h_component, color=COLORS['component'], linestyle='--', linewidth=4, 
                label=f'Component (h={h_component:.3f}, n={n_component})', alpha=0.9, zorder=10)
    
    ax1.set_yscale('symlog', linthresh=0.1)
    ax1.set_ylim(0, max(h_activity * 1.5, max_height * 0.5))
    ax1.set_xlabel('Atomic Segment Index', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Linkage Distance (symlog scale)', fontsize=13, fontweight='bold')
    ax1.set_title('Hierarchical Clustering Dendrogram', fontsize=16, fontweight='bold', pad=15)
    ax1.legend(fontsize=12, loc='upper right', framealpha=0.95)
    ax1.grid(True, alpha=0.3, axis='y', linestyle=':', which='both')
    ax1.set_facecolor('#F8F9FA')
    
    # ========== Subplot 2: Timeline Comparison ==========
    ax2 = fig.add_subplot(gs[1, :])
    
    # Calculate event boundaries for each level
    def get_boundaries(labels):
        boundaries = [0]
        for i in range(1, len(labels)):
            if labels[i] != labels[i-1]:
                boundaries.append(i)
        boundaries.append(len(labels))
        return boundaries
    
    bounds_activity = get_boundaries(labels_activity)
    bounds_side = get_boundaries(labels_side)
    bounds_component = get_boundaries(labels_component)
    
    # Plot timelines
    y_positions = [2, 1, 0]
    all_bounds = [bounds_activity, bounds_side, bounds_component]
    all_colors = [COLORS['activity'], COLORS['side'], COLORS['component']]
    all_labels_list = [labels_activity, labels_side, labels_component]
    level_names = ['Activity', 'Side', 'Component']
    
    colors_map = plt.cm.Set3
    
    for y_pos, bounds, color, labels_list, name in zip(y_positions, all_bounds, all_colors, all_labels_list, level_names):
        for i in range(len(bounds) - 1):
            start = bounds[i]
            end = bounds[i + 1]
            event_id = labels_list[start]
            
            rect = plt.Rectangle((start, y_pos - 0.35), end - start, 0.7,
                                facecolor=colors_map(event_id % 12), 
                                edgecolor=color, linewidth=2.5, alpha=0.8)
            ax2.add_patch(rect)
            
            # Add event ID if wide enough
            if end - start > 5:
                ax2.text((start + end) / 2, y_pos, str(event_id), 
                        ha='center', va='center', fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    ax2.set_xlim(0, K)
    ax2.set_ylim(-0.5, 2.5)
    ax2.set_yticks(y_positions)
    ax2.set_yticklabels([f'{name} ({len(b)-1})' for name, b in zip(level_names, all_bounds)])
    ax2.set_xlabel('Atomic Segment Index', fontsize=12, fontweight='bold')
    ax2.set_title('Multi-level Timeline Comparison', fontsize=14, fontweight='bold', pad=10)
    ax2.grid(True, alpha=0.2, axis='x')
    ax2.set_facecolor('#F8F9FA')
    
    # ========== Subplot 3: Cluster Statistics ==========
    ax3 = fig.add_subplot(gs[2, 0])
    
    cluster_counts = [n_activity, n_side, n_component]
    bars = ax3.bar(['Activity', 'Side', 'Component'], cluster_counts, 
                   color=[COLORS['activity'], COLORS['side'], COLORS['component']], 
                   alpha=0.7, edgecolor='black', linewidth=2)
    
    for bar, count in zip(bars, cluster_counts):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}', ha='center', va='bottom', fontsize=14, fontweight='bold')
    
    ax3.set_ylabel('Number of Clusters', fontsize=12, fontweight='bold')
    ax3.set_title('Cluster Count by Level', fontsize=14, fontweight='bold', pad=10)
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.set_facecolor('#F8F9FA')
    
    # ========== Subplot 4: Cut Heights ==========
    ax4 = fig.add_subplot(gs[2, 1])
    
    heights = [h_activity, h_side, h_component]
    bars = ax4.bar(['Activity', 'Side', 'Component'], heights,
                   color=[COLORS['activity'], COLORS['side'], COLORS['component']], 
                   alpha=0.7, edgecolor='black', linewidth=2)
    
    for bar, h in zip(bars, heights):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{h:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax4.set_ylabel('Cut Height (h)', fontsize=12, fontweight='bold')
    ax4.set_title('Cut Heights by Level', fontsize=14, fontweight='bold', pad=10)
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.set_facecolor('#F8F9FA')
    
    plt.tight_layout()
    plt.show()
    
    # ========== Print detailed timeline information ==========
    print('\n' + '='*80)
    print('📅 DETAILED EVENT TIMELINE')
    print('='*80)
    
    def format_time(frames):
        """Convert frames to MM:SS format"""
        seconds = frames / FPS
        mins = int(seconds // 60)
        secs = int(seconds % 60)
        return f'{mins:02d}:{secs:02d}'
    
    def print_level_timeline(level_name, labels, color_code):
        """Print timeline for one level"""
        print(f'\n{color_code}▶ {level_name} Level{chr(27)}[0m')
        print('-' * 80)
        
        # Get boundaries
        boundaries = [0]
        for i in range(1, len(labels)):
            if labels[i] != labels[i-1]:
                boundaries.append(i)
        boundaries.append(len(labels))
        
        n_events = len(boundaries) - 1
        print(f'Total Events: {n_events}\n')
        
        # Print each event
        for i in range(n_events):
            start_idx = boundaries[i]
            end_idx = boundaries[i + 1]
            event_id = labels[start_idx]
            
            # Get time range from segs (in frames)
            start_frame = segs[start_idx][0]  # First segment's start frame
            end_frame = segs[end_idx - 1][1]  # Last segment's end frame
            duration_frames = end_frame - start_frame
            duration = duration_frames / FPS  # Convert to seconds
            
            # Number of atomic segments
            n_segments = end_idx - start_idx
            
            print(f'  Event {event_id:2d}: {format_time(start_frame)} → {format_time(end_frame)} '
                  f'(Duration: {duration:6.1f}s, {n_segments:3d} segments)')
        
        print()
    
    # Print timelines for all three levels
    print_level_timeline('🔴 Activity', labels_activity, '\033[91m')
    print_level_timeline('🟠 Side', labels_side, '\033[93m')
    print_level_timeline('🟢 Component', labels_component, '\033[92m')
    
    print('='*80)
    print(f'Total video duration: {format_time(segs[-1][1])}')
    print('='*80 + '\n')

print('✅ Visualization function defined')

✅ Visualization function defined


## 🔗 Connect Widgets to Visualization

In [7]:
output = widgets.Output()
timeline_output = widgets.Output()  # Separate output for timeline table

def generate_timeline_table(h_activity, h_side, h_component):
    """Generate HTML table with detailed timeline"""
    from IPython.display import HTML
    
    # Get cluster labels
    labels_activity = fcluster(Z, t=h_activity, criterion='distance')
    labels_side = fcluster(Z, t=h_side, criterion='distance')
    labels_component = fcluster(Z, t=h_component, criterion='distance')
    
    def format_time(frames):
        seconds = frames / FPS
        mins = int(seconds // 60)
        secs = int(seconds % 60)
        return f'{mins:02d}:{secs:02d}'
    
    def get_events_info(labels):
        boundaries = [0]
        for i in range(1, len(labels)):
            if labels[i] != labels[i-1]:
                boundaries.append(i)
        boundaries.append(len(labels))
        
        events = []
        for i in range(len(boundaries) - 1):
            start_idx = boundaries[i]
            end_idx = boundaries[i + 1]
            event_id = labels[start_idx]
            start_frame = segs[start_idx][0]
            end_frame = segs[end_idx - 1][1]
            duration_frames = end_frame - start_frame
            duration = duration_frames / FPS  # Convert to seconds
            n_segments = end_idx - start_idx
            
            events.append({
                'id': event_id,
                'start_str': format_time(start_frame),
                'end_str': format_time(end_frame),
                'duration': duration,
                'segments': n_segments
            })
        return events
    
    # Get events for all levels
    activity_events = get_events_info(labels_activity)
    side_events = get_events_info(labels_side)
    component_events = get_events_info(labels_component)
    
    # Build HTML table
    html = '<div style="max-height: 500px; overflow-y: auto; font-family: Arial, sans-serif;">'
    
    def make_table(events, title, color):
        table_html = f'<h3 style="color: {color}; margin-top: 15px;">{title} ({len(events)} events)</h3>'
        table_html += '<table style="width: 100%; border-collapse: collapse; font-size: 13px; margin-bottom: 20px;">'
        table_html += '<tr style="background-color: #f0f0f0; font-weight: bold;">'
        table_html += '<th style="border: 1px solid #ddd; padding: 10px; width: 15%;">Event ID</th>'
        table_html += '<th style="border: 1px solid #ddd; padding: 10px; width: 20%;">Start Time</th>'
        table_html += '<th style="border: 1px solid #ddd; padding: 10px; width: 20%;">End Time</th>'
        table_html += '<th style="border: 1px solid #ddd; padding: 10px; width: 20%;">Duration (s)</th>'
        table_html += '<th style="border: 1px solid #ddd; padding: 10px; width: 25%;">Atomic Segments</th>'
        table_html += '</tr>'
        
        for e in events:
            table_html += '<tr style="background-color: white;">'
            table_html += f'<td style="border: 1px solid #ddd; padding: 8px; text-align: center; font-weight: bold;">{e["id"]}</td>'
            table_html += f'<td style="border: 1px solid #ddd; padding: 8px; text-align: center;">{e["start_str"]}</td>'
            table_html += f'<td style="border: 1px solid #ddd; padding: 8px; text-align: center;">{e["end_str"]}</td>'
            table_html += f'<td style="border: 1px solid #ddd; padding: 8px; text-align: center;">{e["duration"]:.1f}</td>'
            table_html += f'<td style="border: 1px solid #ddd; padding: 8px; text-align: center;">{e["segments"]}</td>'
            table_html += '</tr>'
        table_html += '</table>'
        return table_html
    
    html += make_table(activity_events, '🔴 Activity Level', '#E74C3C')
    html += make_table(side_events, '🟠 Side Level', '#F39C12')
    html += make_table(component_events, '🟢 Component Level', '#27AE60')
    html += '</div>'
    
    return HTML(html)

def on_slider_change(change):
    """Called when any slider changes"""
    # Update slider constraints to maintain hierarchy
    slider_side.max = slider_activity.value - 0.1
    slider_component.max = slider_side.value - 0.05
    slider_activity.min = slider_side.value + 0.1
    slider_side.min = slider_component.value + 0.05
    
    with output:
        clear_output(wait=True)
        update_visualization(slider_activity.value, slider_side.value, slider_component.value)
    
    # Update timeline table
    with timeline_output:
        clear_output(wait=True)
        html_table = generate_timeline_table(slider_activity.value, slider_side.value, slider_component.value)
        display(html_table)

def on_reset_click(b):
    """Reset to initial values"""
    slider_activity.value = initial_h_activity
    slider_side.value = initial_h_side
    slider_component.value = initial_h_component

def on_auto_click(b):
    """Auto-optimize cut heights"""
    with output:
        clear_output(wait=True)
        optimal_h, scores = auto_optimize_cuts(Z, distance_matrix, n_levels=3)
        
        if len(optimal_h) >= 3:
            slider_activity.value = optimal_h[0]
            slider_side.value = optimal_h[1]
            slider_component.value = optimal_h[2]
        
        update_visualization(slider_activity.value, slider_side.value, slider_component.value)

# Attach event handlers
slider_activity.observe(on_slider_change, names='value')
slider_side.observe(on_slider_change, names='value')
slider_component.observe(on_slider_change, names='value')
reset_button.on_click(on_reset_click)
auto_button.on_click(on_auto_click)

print('✅ Event handlers connected')

✅ Event handlers connected


## 🚀 Launch Interactive Tool

In [8]:
# Create UI layout
ui = widgets.VBox([
    widgets.HTML('<h2>🎛️ Interactive Clustering Control Panel</h2>'),
    widgets.HTML('<p>Adjust the sliders to change cut heights. The visualization and timeline will update automatically.</p>'),
    widgets.HTML('<hr>'),
    widgets.HBox([slider_activity, label_activity]),
    widgets.HBox([slider_side, label_side]),
    widgets.HBox([slider_component, label_component]),
    widgets.HTML('<hr>'),
    widgets.HBox([reset_button, auto_button]),
    widgets.HTML('<hr>'),
    widgets.HTML('<h3>📊 Visualizations</h3>'),
    output,
    widgets.HTML('<hr style="margin-top: 30px;">'),
    widgets.HTML('<h3>📅 Detailed Event Timeline</h3>'),
    timeline_output
])

# Display UI
display(ui)

# Initial visualization and timeline
with output:
    update_visualization(initial_h_activity, initial_h_side, initial_h_component)

with timeline_output:
    html_table = generate_timeline_table(initial_h_activity, initial_h_side, initial_h_component)
    display(html_table)

print('\n✅ Interactive tool launched!')
print('\n📌 Instructions:')
print('  1. Drag the sliders to adjust cut heights')
print('  2. Click "Auto-Optimize" to find optimal cuts automatically')
print('  3. Click "Reset to Initial" to restore original values')
print('  4. Watch the dendrogram, timeline, and event table update in real-time!')
print('  5. Scroll down to see the detailed event timeline with timestamps!')


✅ Interactive tool launched!

📌 Instructions:
  1. Drag the sliders to adjust cut heights
  2. Click "Auto-Optimize" to find optimal cuts automatically
  3. Click "Reset to Initial" to restore original values
  4. Watch the dendrogram, timeline, and event table update in real-time!
  5. Scroll down to see the detailed event timeline with timestamps!
